<a href="https://colab.research.google.com/github/ThLanBot42/DHU_AI_THL_CV_COMP_COMPETITION/blob/main/cv_comp_V1.3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU 可用: {torch.cuda.get_device_name(0)}")
else:
    print("GPU 不可用，使用 CPU")

GPU 可用: Tesla T4


In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.4 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
print("模型加载成功")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
模型加载成功


In [3]:
import os

base = '/content/datasets/wheat'

folders = [
    'images/train',
    'images/val',
    'images/test',
    'labels/train',
    'labels/val',
    'labels/test'
]

for f in folders:
    os.makedirs(os.path.join(base, f), exist_ok=True)

print("文件夹创建完成")


文件夹创建完成


In [ ]:
data_yaml = """
train: /content/datasets/wheat/images/train
val: /content/datasets/wheat/images/val
test: /content/datasets/wheat/images/test

nc: 1
names: ['wheat']
"""

with open('/content/datasets/wheat/data.yaml', 'w') as f:
    f.write(data_yaml)

print("data.yaml 创建完成")

data.yaml 创建完成


此处发现没有标签不能直接进行训练，所以采取方案一“伪标签法”，即在此环境下采取“学霸带学渣”的方式完成标签的产生。

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from ultralytics import YOLO
import os

# 1.加载模型
model = YOLO('yolov8x.pt')

# 2.定义图片路径
# 注意：请确保路径与你之前的目录结构一致
source_path = '/content/drive/MyDrive/AI_competition/images/train'

# 3.执行预测并自动保存为 YOLO 格式的 txt
# save_txt=True 会在 runs/detect/predict/labels 下生成 txt
# conf=0.25 是置信度阈值，你可以根据生成效果调整
results = model.predict(source=source_path, save_txt=True, conf=0.2, device=0)

print("✅ 伪标签生成完成！请去 'runs/detect/predict/labels' 文件夹查看生成的 txt 文件。")


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/4116 /content/drive/MyDrive/AI_competition/images/train/0007634580386bd39d4d0d24df58893c3bb967e12d6fc065ce8659e9acacc928.png: 640x640 (no detections), 96.6ms
image 2/4116 /content/drive/MyDrive/AI_competition/images/train/00319488e879a811698174d9f26ef174f2f108a13e12edee5a3c50899ed26336.png: 640x640 (no detections), 82.1ms
image 3/4116 /content/drive/MyDrive/AI_competition/images/train/00417fc37a35f14ce3044bf2ae6c76d19bf42a426a94038e9f96dfe5fe8a25

In [ ]:

# 修改 predict 调用，指定保存路径到 Drive
results = model.predict(
    source=source_path,
    save_txt=True,
    conf=0.2,
    device=0,
    project="/content/drive/MyDrive/AI_competition/",  # 项目根目录
    name="pseudo_labels_train"  # 本次预测的文件夹名
)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/4116 /content/drive/MyDrive/AI_competition/images/train/0007634580386bd39d4d0d24df58893c3bb967e12d6fc065ce8659e9acacc928.png: 640x640 (no detections), 52.7ms
image 2/4116 /content/drive/MyDrive/AI_competition/images/train/00319488e879a811698174d9f26ef174f2f108a13e12edee5a3c50899ed26336.png: 640x640 (no detections), 58.4ms
image 3/4116 /content/drive/MyDrive/AI_competition/images/train/00417fc37a35f14ce3044bf2ae6c76d19bf42a426a94038e9f96dfe5fe8a25

In [6]:
import os
import shutil

# 修正后的路径：加上了 AI_competition
drive_labels_path = '/content/drive/MyDrive/AI_competition/pseudo_labels_train/labels'
local_labels_path = '/content/datasets/wheat/labels/train'

os.makedirs(local_labels_path, exist_ok=True)

if os.path.exists(drive_labels_path):
    files = [f for f in os.listdir(drive_labels_path) if f.endswith('.txt')]
    for f in files:
        shutil.copy(os.path.join(drive_labels_path, f), os.path.join(local_labels_path, f))
    print(f"✅ 成功同步 {len(files)} 个标签文件至本地训练目录！")
else:
    print(f"❌ 还是找不到文件夹！请手动检查云端硬盘中 AI_competition/pseudo_labels_train 文件夹是否存在。")

✅ 成功同步 1208 个标签文件至本地训练目录！


In [8]:
import os

# 1. 重新定义并创建所有文件夹（防止丢失）
base = '/content/datasets/wheat'
for f in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    os.makedirs(os.path.join(base, f), exist_ok=True)

# 2. 【关键】建立图片链接（把云端硬盘的图片“映射”到本地，不需要重新下载）
# 请确认您的图片路径是这个，如果不对请微调
drive_images = '/content/drive/MyDrive/AI_competition/images/train'
local_images = '/content/datasets/wheat/images/train'

if not os.path.exists(local_images + '/0a938aa9dd7a40e058200c22cbcd5a3225c30dd5dcee34949efc9d382cf7d7bf.jpg'): # 检查是否已链接
    !ln -s {drive_images}/* {local_images}/
    print("✅ 图片映射完成")

# 3. 重新生成 data.yaml（确保文件一定存在）
data_yaml_content = f"""
path: {base}
train: images/train
val: images/train
test: images/train

nc: 1
names: ['wheat']
"""

with open(os.path.join(base, 'data.yaml'), 'w') as f:
    f.write(data_yaml_content)

if os.path.exists(os.path.join(base, 'data.yaml')):
    print(f"✅ data.yaml 已成功创建在: {os.path.join(base, 'data.yaml')}")
else:
    print("❌ data.yaml 创建失败，请检查权限。")

✅ 图片映射完成
✅ data.yaml 已成功创建在: /content/datasets/wheat/data.yaml


In [9]:
from ultralytics import YOLO

# 加载模型
model = YOLO('yolov8m.pt')

# 开始训练
model.train(
    data='/content/datasets/wheat/data.yaml',
    epochs=30,
    imgsz=640,
    batch=32,
    device=0,
    project='/content/drive/MyDrive/AI_competition/training_runs',
    name='wheat_v1_baseline'
)

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/wheat/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=wheat_v1_baseline-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e28c3fbc170>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [1]:
import os
import shutil

drive_labels_path = '/content/drive/MyDrive/AI_competition/pseudo_labels_train/labels'
local_labels_path = '/content/datasets/wheat/labels/train'

os.makedirs(local_labels_path, exist_ok=True)

files = [f for f in os.listdir(drive_labels_path) if f.endswith('.txt')]
for f in files:
    shutil.copy(os.path.join(drive_labels_path, f), os.path.join(local_labels_path, f))

print(f"✅ 成功同步 {len(files)} 个训练集标签文件至本地！")

✅ 成功同步 1208 个训练集标签文件至本地！


In [2]:
import os

local_labels_path = '/content/datasets/wheat/labels/train'
count = 0

# 遍历所有标签文件进行检查与修复
for filename in os.listdir(local_labels_path):
    if filename.endswith('.txt'):
        filepath = os.path.join(local_labels_path, filename)

        with open(filepath, 'r') as f:
            lines = f.readlines()

        new_lines = []
        for line in lines:
            parts = line.strip().split()
            # 确保是有效的 YOLO 格式
            if len(parts) >= 5:
                parts[0] = '0'  # 强制类别索引为 0
                new_lines.append(' '.join(parts) + '\n')

        with open(filepath, 'w') as f:
            f.writelines(new_lines)
            count += 1

print(f"🛠 校验完成，已成功修复 {count} 个标签文件的类别索引！")

🛠 校验完成，已成功修复 1208 个标签文件的类别索引！


In [3]:
import os

label_path = '/content/datasets/wheat/labels/train'

files = os.listdir(label_path)
print("标签文件数量:", len(files))

# 看前3个文件内容
for f in files[:3]:
    print("\n文件:", f)
    with open(os.path.join(label_path, f)) as fp:
        print(fp.read())

标签文件数量: 1208

文件: 351eb6a2f03e19e5096087f863c2f6c5a9994e9b7e5e7caa33960cf0639035f3.txt
0 0.11041 0.0577719 0.219887 0.114625


文件: 732913e6264c355bbb1ab15929ebe8c4117957843770fec3da00032b71b4fb62.txt
0 0.808229 0.370701 0.0674326 0.0828263
0 0.477886 0.236477 0.0879312 0.0955287
0 0.510712 0.337746 0.310839 0.300453
0 0.844795 0.0937257 0.133297 0.149295
0 0.703417 0.8418 0.0464395 0.077306


文件: 681fabac8e6989cbb3dfe0bc97919a0780180bf6e6b6a20a041d56df4a925d95.txt
0 0.88376 0.535915 0.071537 0.0560139
0 0.780935 0.427939 0.192909 0.054668
0 0.958515 0.318592 0.0613933 0.0719015
0 0.930205 0.595835 0.139158 0.0800347
0 0.952021 0.829622 0.095145 0.33822
0 0.952356 0.750393 0.0946072 0.178531
0 0.321581 0.152262 0.0557907 0.0736149



In [6]:
import os, shutil, random

train_img = '/content/drive/MyDrive/AI_competition/images/train'
train_lbl = '/content/drive/MyDrive/AI_competition/labels/train'

val_img = '/content/drive/MyDrive/AI_competition/images/val'
val_lbl = '/content/drive/MyDrive/AI_competition/labels/val'

os.makedirs(val_img, exist_ok=True)
os.makedirs(val_lbl, exist_ok=True)

files = os.listdir(train_img)
random.shuffle(files)

val_num = int(len(files) * 0.1)

for f in files[:val_num]:
    # 移动图片
    shutil.move(os.path.join(train_img, f), os.path.join(val_img, f))

    # ⭐关键修复
    label_name = os.path.splitext(f)[0] + '.txt'

    src_label = os.path.join(train_lbl, label_name)
    dst_label = os.path.join(val_lbl, label_name)

    # 防止标签不存在报错
    if os.path.exists(src_label):
        shutil.move(src_label, dst_label)
    else:
        print("⚠️ 找不到标签:", label_name)

print("✅ val 集创建完成")

⚠️ 找不到标签: 00b984a8a9c1e7a275f53bc98f32b5ce980fe7b8acb484813487d081c1a76f19.txt
⚠️ 找不到标签: b6b3bb88d7b7ed633301fa66770afc6caecd4276b251c4685d7bdd5696840bfc.txt
⚠️ 找不到标签: 417dfee4bfd09256f2b011a394a742d90fbfb124453ca449f89a29a93bf47c35.txt
⚠️ 找不到标签: 40d061e119178bec3a2da27bdd3efa057cff55442980576d9f38bee3aa33b8ce.txt
⚠️ 找不到标签: 2886cdad46221c54b578822571f5923e60357f1eda669308fe6ea644de76d188.txt
⚠️ 找不到标签: 2f032ae2d8118e1afda9c9d6a2b6abd3cf23355d88d7085fd058c0394c7167d4.txt
⚠️ 找不到标签: 06e6695fc3baf867c141a6692df8f60092648752fee36f0b9b301732f229d3d1.txt
⚠️ 找不到标签: 2ec32dacaf5e2f816b124f235a2eb20dd9a31f6e5d32332b224a0ea3bb881347.txt
⚠️ 找不到标签: 50844a14e09b5813e7beb167047434ebdaba8a53c9d1efb6573d1f7992b5319d.txt
⚠️ 找不到标签: 747d421f313856016fa30484ed4d9df63b25b42e267e1a81f3be3a6f52cf9faa.txt
⚠️ 找不到标签: 9331cf27d22bb041278d14ccf9d98158589d906342dd47f2d5e09881c43d52e8.txt
⚠️ 找不到标签: 8d0ea3ab51664823a7900bac096be9fb1dcf1ee2529022555f3ae1c4b7760573.txt
⚠️ 找不到标签: bbf62dcb9965f4406bdd50fe28cd521335bcd5c89b

In [7]:
import os

img_dir = '/content/drive/MyDrive/AI_competition/images/train'
lbl_dir = '/content/drive/MyDrive/AI_competition/labels/train'

imgs = os.listdir(img_dir)
labels = os.listdir(lbl_dir)

print("图片数量:", len(imgs))
print("标签数量:", len(labels))
print("有标签比例:", len(labels)/len(imgs))

图片数量: 3704
标签数量: 0
有标签比例: 0.0
